# 03 · Evaluation

We compare the two trained generators on two complementary axes:

1. **Fidelity & diversity** — Frechet Inception Distance (FID) against the real MNIST test set.
2. **Utility for downstream tasks** — train a small CNN on *synthetic-only* data and report its accuracy on the *real* MNIST test set.

We also produce a side-by-side `Real vs cGAN vs Diffusion` grid for a qualitative comparison.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from data.dataloader import get_mnist_loaders
from model.cgan import Generator
from model.diffusion import ConditionalUNet
from training.train_diffusion import (
    linear_beta_schedule, DiffusionSchedule, sample_images,
)
from utils.checkpoint import load_checkpoint
from utils.metrics import compute_fid, InceptionFeatures
from utils.visualize import plot_comparison_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Load the trained models

Replace the checkpoint paths with whichever epoch you trained to.

In [ ]:
NUM_CLASSES = 10
LATENT_DIM = 100
NUM_TIMESTEPS = 1000

G = Generator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(device)
unet = ConditionalUNet(num_classes=NUM_CLASSES, base_channels=64, time_dim=128).to(device)

# Pick the latest checkpoints from each training run
cgan_ckpts = sorted(Path('checkpoints/cgan').glob('cgan_epoch_*.pt'))
diff_ckpts = sorted(Path('checkpoints/diffusion').glob('diffusion_epoch_*.pt'))
print('cgan ckpt :', cgan_ckpts[-1] if cgan_ckpts else 'MISSING')
print('diff ckpt :', diff_ckpts[-1] if diff_ckpts else 'MISSING')

load_checkpoint(cgan_ckpts[-1], models={'generator': G}, map_location=device)
load_checkpoint(diff_ckpts[-1], models={'unet': unet}, map_location=device)
G.eval(); unet.eval()

## Generate large synthetic batches per class

In [ ]:
@torch.no_grad()
def cgan_generate(n_per_class: int = 200) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (N, 1, 28, 28) images and (N,) labels from the cGAN."""
    labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(n_per_class)
    z = torch.randn(labels.size(0), LATENT_DIM, device=device)
    return G(z, labels).cpu(), labels.cpu()

betas = linear_beta_schedule(NUM_TIMESTEPS)
schedule = DiffusionSchedule.from_betas(betas).to(device)

@torch.no_grad()
def diffusion_generate(n_per_class: int = 100, batch: int = 200) -> tuple[torch.Tensor, torch.Tensor]:
    """Return synthetic samples from the diffusion model.

    Diffusion sampling is expensive (T denoising steps per sample), so we
    generate in batches of `batch` images at a time.
    """
    all_imgs, all_labels = [], []
    full_labels = torch.arange(NUM_CLASSES).repeat_interleave(n_per_class)
    for i in range(0, full_labels.size(0), batch):
        chunk = full_labels[i : i + batch].to(device)
        imgs = sample_images(unet, chunk, schedule).cpu()
        all_imgs.append(imgs)
        all_labels.append(chunk.cpu())
    return torch.cat(all_imgs), torch.cat(all_labels)

# Use a smaller diffusion sample count if you're CPU-bound.
cgan_imgs, cgan_labels = cgan_generate(n_per_class=200)
diff_imgs, diff_labels = diffusion_generate(n_per_class=100)
print('cgan samples :', cgan_imgs.shape)
print('diff samples :', diff_imgs.shape)

## Qualitative comparison: Real vs cGAN vs Diffusion

In [ ]:
_, test_loader = get_mnist_loaders(batch_size=512, num_workers=0)
real_imgs, real_labels = next(iter(test_loader))

def first_per_class(images: torch.Tensor, labels: torch.Tensor, classes=range(10)) -> dict[int, torch.Tensor]:
    by_class = {}
    for c in classes:
        mask = labels == c
        if mask.any():
            by_class[c] = images[mask][:1]
    return by_class

panels = {
    'Real':      first_per_class(real_imgs, real_labels),
    'cGAN':      first_per_class(cgan_imgs, cgan_labels),
    'Diffusion': first_per_class(diff_imgs, diff_labels),
}
fig = plot_comparison_grid(panels, classes=range(10), title='Real vs cGAN vs Diffusion')
plt.show()

## Quantitative: FID

We compute FID against a fixed reference sample of real MNIST test images. Lower is better — FID values for MNIST-like models typically sit in the 5–50 range depending on training budget.

In [ ]:
# Build a fixed real reference set the same size as our synthetic batches.
n_ref = min(2000, real_imgs.size(0))
real_ref = real_imgs[:n_ref]
print('real reference:', real_ref.shape)

extractor = InceptionFeatures().to(device).eval()

fid_cgan = compute_fid(real_ref, cgan_imgs, extractor=extractor, device=device, batch_size=64)
fid_diff = compute_fid(real_ref, diff_imgs, extractor=extractor, device=device, batch_size=64)

print(f'FID cGAN      : {fid_cgan:8.3f}')
print(f'FID Diffusion : {fid_diff:8.3f}')

### Analysis

* **Higher fidelity:** the model with the lower FID score produces samples whose Inception features more closely match the real distribution.
* **Diversity:** GANs are known to *mode collapse* (producing too few distinct images per class), which inflates FID even when individual samples look sharp. Diffusion models tend to be more diverse but slower to sample.
* **Practical suitability:** for *large-scale* data generation, cGAN sampling is one forward pass per image while diffusion needs `T` passes — typically two to three orders of magnitude slower. If both FIDs are acceptable, the cGAN is the obvious choice for a CAPTCHA pipeline that needs millions of images.

Fill in the actual observed FID values and the qualitative read of the comparison grid above when reporting results.

## Downstream utility — CNN trained on synthetic, evaluated on real

If the synthetic samples really capture the structure of MNIST, a classifier trained on them alone should still generalize to the real test set. We train one small CNN per synthetic source and report top-1 accuracy on the real MNIST test set.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2), # 7x7
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


def train_classifier(images: torch.Tensor, labels: torch.Tensor,
                    *, epochs: int = 5, batch_size: int = 128) -> SimpleCNN:
    ds = TensorDataset(images, labels)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = SimpleCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(1, epochs + 1):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            total += y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            loss_sum += loss.item() * y.size(0)
        print(f'  ep {ep}: loss={loss_sum / total:.4f} acc={correct / total:.4f}')
    return model


@torch.no_grad()
def evaluate(model: SimpleCNN, loader: DataLoader) -> float:
    model.eval()
    total, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
print('--- training on cGAN samples ---')
clf_cgan = train_classifier(cgan_imgs, cgan_labels, epochs=5)
acc_cgan = evaluate(clf_cgan, test_loader)
print(f'cGAN-trained CNN on real MNIST test: {acc_cgan:.4f}')

print('--- training on Diffusion samples ---')
clf_diff = train_classifier(diff_imgs, diff_labels, epochs=5)
acc_diff = evaluate(clf_diff, test_loader)
print(f'Diffusion-trained CNN on real MNIST test: {acc_diff:.4f}')

## Summary

| Metric                         | cGAN | Diffusion |
|--------------------------------|------|-----------|
| FID (lower better)             | …    | …         |
| Downstream test accuracy       | …    | …         |
| Sampling cost / image          | 1 forward pass | T forward passes |

Fill in the numeric cells with your run's results, then decide which model best fits SuperCognition's CAPTCHA pipeline: typically the cGAN wins on throughput while diffusion wins on diversity. For an operational system you would combine both — diffusion for a curated high-quality seed set, cGAN for bulk augmentation.

## Optional stand-out extension: multi-digit CAPTCHA composition

Stitch four single-digit samples together into a CAPTCHA-style image.

In [ ]:
def make_captcha(digits: str, source: str = 'cgan') -> torch.Tensor:
    labels = torch.tensor([int(c) for c in digits], device=device)
    if source == 'cgan':
        with torch.no_grad():
            z = torch.randn(labels.size(0), LATENT_DIM, device=device)
            imgs = G(z, labels)
    elif source == 'diffusion':
        imgs = sample_images(unet, labels, schedule)
    else:
        raise ValueError(source)
    return torch.cat([img for img in imgs.squeeze(1)], dim=1)  # concatenate horizontally

captcha = make_captcha('7392', source='cgan')
plt.figure(figsize=(6, 2))
plt.imshow(((captcha.clamp(-1, 1) + 1) / 2).cpu(), cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('cGAN CAPTCHA: 7392')
plt.show()